# Operational Alert Panel

This notebook turns the benchmark and latest forecasts into an operational early-warning view for Albanian cities.


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown


In [3]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ALERT_DIR = PROJECT_ROOT / "outputs" / "operational_alerts"

daily = pd.read_csv(PROCESSED_DIR / "albania_air_quality_daily.csv", parse_dates=["date"])
alerts = pd.read_csv(ALERT_DIR / "forecast_alerts.csv", parse_dates=["forecast_date", "latest_observed_date"])
ranking = pd.read_csv(ALERT_DIR / "city_attention_ranking.csv", parse_dates=["peak_risk_date"])
heatmap = pd.read_csv(ALERT_DIR / "alert_heatmap_table.csv", parse_dates=["forecast_date"])

display(Markdown(
    f"Loaded **{len(alerts)}** forecast-alert rows across **{ranking['city'].nunique()}** Albanian cities."
))
ranking.head()


Loaded **24** forecast-alert rows across **8** Albanian cities.

,city,max_alert_score,peak_predicted_aqi,peak_predicted_pm25,max_aqi_change,max_pm25_change,peak_risk_date,peak_horizon_days,predicted_aqi_label,final_alert_level,recommended_action,selected_aqi_model,selected_pm2_5_model,selected_aqi_source,selected_pm2_5_source
0,Tirane,3,48.000000,14.191430,0.000000,-0.962737,2026-05-01,3,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,HistGBM Global Features,enhanced,enhanced
1,Fier,3,47.000000,12.890626,0.000000,-0.701041,2026-04-30,2,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,ARIMA,enhanced,baseline
2,Vlore,3,47.000000,12.020833,0.000000,0.000000,2026-04-30,2,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,Persistence Current,enhanced,enhanced
3,Berat,3,47.000000,11.550555,0.000000,0.233889,2026-04-29,1,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,HistGBM Global Features,enhanced,enhanced
4,Durres,3,45.768741,14.266126,-7.231259,-1.096374,2026-04-29,1,Moderate,Warning,Prepare local warning communication and monito...,HistGBM Global Features,ARIMA,enhanced,baseline


In [4]:
alert_colors = {
    "Stable": "#1f77b4",
    "Watch": "#2ca02c",
    "Warning": "#ffbf00",
    "High": "#ff7f0e",
    "Severe": "#d62728",
    "Critical": "#7f0000",
}

city_widget = widgets.Dropdown(
    options=ranking['city'].tolist(),
    value=ranking['city'].iloc[0],
    description='City:',
    layout=widgets.Layout(width='220px'),
)

history_widget = widgets.IntSlider(
    value=30,
    min=14,
    max=90,
    step=2,
    description='History:',
    continuous_update=False,
)

output = widgets.Output()


In [5]:
def render_panel(*_):
    output.clear_output(wait=True)
    city = city_widget.value
    history_days = history_widget.value

    ranking_order = ranking.sort_values(['max_alert_score', 'peak_predicted_aqi', 'peak_predicted_pm25'], ascending=[False, False, False])['city'].tolist()
    city_rank = ranking[ranking['city'] == city].iloc[0]
    city_alerts = alerts[alerts['city'] == city].sort_values('forecast_date').copy()

    ranking_fig = px.bar(
        ranking,
        x='city',
        y='max_alert_score',
        color='final_alert_level',
        color_discrete_map=alert_colors,
        category_orders={'city': ranking_order},
        template='plotly_white',
        title='City attention ranking for the next 1-3 days',
        hover_data=['peak_predicted_aqi', 'peak_predicted_pm25', 'peak_risk_date', 'peak_horizon_days'],
    )
    ranking_fig.update_layout(height=420, xaxis_title='City', yaxis_title='Max alert score', legend_title_text='Alert level')

    heat = heatmap.copy()
    heat['forecast_date_str'] = heat['forecast_date'].dt.strftime('%Y-%m-%d')
    heat['hover'] = (
        'City: ' + heat['city'] + '<br>Date: ' + heat['forecast_date_str'] +
        '<br>Alert: ' + heat['final_alert_level'] +
        '<br>AQI: ' + heat['predicted_european_aqi_max'].round(2).astype(str)
    )
    heat_pivot = heat.pivot(index='city', columns='forecast_date_str', values='alert_score').reindex(ranking_order)
    hover_pivot = heat.pivot(index='city', columns='forecast_date_str', values='hover').reindex(ranking_order)
    heatmap_fig = go.Figure(
        data=go.Heatmap(
            z=heat_pivot.values,
            x=list(heat_pivot.columns),
            y=list(heat_pivot.index),
            text=hover_pivot.values,
            hoverinfo='text',
            colorscale=[
                [0.0, '#1f77b4'],
                [0.2, '#2ca02c'],
                [0.4, '#ffbf00'],
                [0.6, '#ff7f0e'],
                [0.8, '#d62728'],
                [1.0, '#7f0000'],
            ],
            zmin=1,
            zmax=6,
            colorbar=dict(title='Alert score'),
        )
    )
    heatmap_fig.update_layout(template='plotly_white', height=420, title='Alert heatmap by city and forecast date')

    city_history = daily[daily['city'] == city].sort_values('date').tail(history_days)
    aqi_fig = go.Figure()
    aqi_fig.add_trace(go.Scatter(x=city_history['date'], y=city_history['european_aqi_max'], mode='lines', name='Observed AQI', line=dict(color='#1f77b4', width=3)))
    aqi_fig.add_trace(go.Scatter(x=city_alerts['forecast_date'], y=city_alerts['predicted_european_aqi_max'], mode='lines+markers', name='Forecast AQI', line=dict(color='#d62728', width=3)))
    aqi_fig.update_layout(template='plotly_white', height=380, title=f'{city} - daily max AQI', xaxis_title='Date', yaxis_title='AQI')

    pm25_fig = go.Figure()
    pm25_fig.add_trace(go.Scatter(x=city_history['date'], y=city_history['pm2_5_mean'], mode='lines', name='Observed PM2.5', line=dict(color='#2ca02c', width=3)))
    pm25_fig.add_trace(go.Scatter(x=city_alerts['forecast_date'], y=city_alerts['predicted_pm2_5_mean'], mode='lines+markers', name='Forecast PM2.5', line=dict(color='#ff7f0e', width=3)))
    pm25_fig.update_layout(template='plotly_white', height=380, title=f'{city} - daily mean PM2.5', xaxis_title='Date', yaxis_title='PM2.5')

    with output:
        display(Markdown(
            f"## Operational Summary - {city}\n"
            f"- Highest alert in the next 1-3 days: **{city_rank['final_alert_level']}**\n"
            f"- Peak risk date: **{city_rank['peak_risk_date'].date()}**\n"
            f"- Peak predicted AQI: **{city_rank['peak_predicted_aqi']:.2f}** ({city_rank['predicted_aqi_label']})\n"
            f"- Peak predicted PM2.5: **{city_rank['peak_predicted_pm25']:.2f}**\n"
            f"- AQI forecast source/model: **{city_rank['selected_aqi_source']} / {city_rank['selected_aqi_model']}**\n"
            f"- PM2.5 forecast source/model: **{city_rank['selected_pm2_5_source']} / {city_rank['selected_pm2_5_model']}**\n"
            f"- Recommended action: **{city_rank['recommended_action']}**"
        ))
        display(ranking_fig)
        display(heatmap_fig)
        display(aqi_fig)
        display(pm25_fig)
        display(Markdown(f'### Selected forecasts and chosen models - {city}'))
        display(city_alerts[['forecast_date', 'horizon_days', 'predicted_european_aqi_max', 'predicted_aqi_label', 'predicted_pm2_5_mean', 'pm25_signal', 'final_alert_level', 'selected_aqi_source', 'selected_aqi_model', 'selected_pm2_5_source', 'selected_pm2_5_model']].reset_index(drop=True))

for widget in [city_widget, history_widget]:
    widget.observe(render_panel, names='value')

display(widgets.HBox([city_widget, history_widget]))
display(output)
render_panel()


Output()

In [6]:
ranking


,city,max_alert_score,peak_predicted_aqi,peak_predicted_pm25,max_aqi_change,max_pm25_change,peak_risk_date,peak_horizon_days,predicted_aqi_label,final_alert_level,recommended_action,selected_aqi_model,selected_pm2_5_model,selected_aqi_source,selected_pm2_5_source
0,Tirane,3,48.000000,14.191430,0.000000,-0.962737,2026-05-01,3,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,HistGBM Global Features,enhanced,enhanced
1,Fier,3,47.000000,12.890626,0.000000,-0.701041,2026-04-30,2,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,ARIMA,enhanced,baseline
2,Vlore,3,47.000000,12.020833,0.000000,0.000000,2026-04-30,2,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,Persistence Current,enhanced,enhanced
3,Berat,3,47.000000,11.550555,0.000000,0.233889,2026-04-29,1,Moderate,Warning,Prepare local warning communication and monito...,Persistence Current,HistGBM Global Features,enhanced,enhanced
4,Durres,3,45.768741,14.266126,-7.231259,-1.096374,2026-04-29,1,Moderate,Warning,Prepare local warning communication and monito...,HistGBM Global Features,ARIMA,enhanced,baseline
5,Shkoder,3,43.259262,11.482311,-5.740738,0.369811,2026-04-29,1,Moderate,Warning,Prepare local warning communication and monito...,HistGBM Global Features,HistGBM Global Features,enhanced,enhanced
6,Elbasan,3,42.892408,11.650258,-1.107592,-0.291409,2026-04-29,1,Moderate,Warning,Prepare local warning communication and monito...,HistGBM Global Features,HistGBM Global Features,enhanced,enhanced
7,Korce,2,40.000000,10.713541,0.000000,0.063541,2026-05-01,3,Fair,Watch,Monitor trends and review the next daily update.,Persistence Current,Ridge Global Features,enhanced,enhanced
